# 🛡️ Drishti-Kavach: Universal Dual BiSeNetV2 Segmentation Training (Kaggle)

This notebook trains a **Universal Dual Railway Segmentation Model** that achieves state-of-the-art segmentation across **BOTH**:
1. 🚂 **Locomotive Cab Windshield Ground-Level Views** (RailSem19)
2. 🇮🇳 **Indian Railways Track, Station & Aerial Drone Views** (UAV-RSOD V1)

### Strategy: Joint Mixed Transfer Learning (10 Epochs)
- Initialized from: `best_bisenetv2_raildrishti.pth` (20-epoch base checkpoint)
- Mixed Batching: 75% Locomotive Cab Frames + 25% Indian Railway Frames per batch
- Dual Validation: Evaluates both domains simultaneously to prevent Catastrophic Forgetting


In [ ]:
# Cell 1: Environment & GPU Verification
import os, sys, time, glob, shutil, random
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[+] Execution Device:', device)
if torch.cuda.is_available():
    print('[+] GPU Name:        ', torch.cuda.get_device_name(0))
    print('[+] GPU Count:       ', torch.cuda.device_count())


In [ ]:
# Cell 2: Dual Dataset & Base Weights Discovery
print('[*] Locating Datasets and Checkpoints...')

# 1. Locate RailSem19 Dataset
rs19_roots = glob.glob('/kaggle/input/**/dataset_segmentation', recursive=True)
RS19_DIR = rs19_roots[0] if rs19_roots else 'dataset_segmentation'
print(f'[+] RailSem19 Path:  {RS19_DIR}')

# 2. Locate UAV-RSOD V1 Dataset
uav_roots = glob.glob('/kaggle/input/**/dataset_segmentation_uav_v1', recursive=True)
uav_zips = glob.glob('/kaggle/input/**/raildrishti_uav*.zip', recursive=True)
if uav_roots:
    UAV_DIR = uav_roots[0]
elif uav_zips:
    import zipfile
    os.makedirs('/kaggle/temp_uav', exist_ok=True)
    with zipfile.ZipFile(uav_zips[0], 'r') as zf:
        zf.extractall('/kaggle/temp_uav')
    found = glob.glob('/kaggle/temp_uav/**/dataset_segmentation_uav_v1', recursive=True)
    UAV_DIR = found[0] if found else '/kaggle/temp_uav/dataset_segmentation_uav_v1'
else:
    UAV_DIR = 'dataset_segmentation_uav_v1'
print(f'[+] UAV-RSOD V1 Path: {UAV_DIR}')

# 3. Locate Base Weights Checkpoint
pth_matches = glob.glob('/kaggle/input/**/best_bisenetv2_raildrishti.pth', recursive=True)
BASE_PTH = pth_matches[0] if pth_matches else 'models/best_bisenetv2_raildrishti.pth'
print(f'[+] Base Checkpoint: {BASE_PTH}')


In [ ]:
# Cell 3: Exact Verified BiSeNetV2 Architecture
"""
BiSeNet V2: Bilateral Network with Guided Aggregation for Real-Time Semantic Segmentation
Implementation tailored for Drishti-Kavach Railway Track & Rail Line Perception.

Reference:
  Yu et al., "BiSeNet V2: Bilateral Network with Guided Aggregation for Real-Time Semantic Segmentation", IJCV 2021.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBNReLU(nn.Module):
    def __init__(self, in_chan, out_chan, ks=3, stride=1, padding=1, dilation=1, groups=1, bias=False):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
            in_chan, out_chan, kernel_size=ks, stride=stride,
            padding=padding, dilation=dilation, groups=groups, bias=bias
        )
        self.bn = nn.BatchNorm2d(out_chan)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class DetailBranch(nn.Module):
    """
    Detail Branch: High spatial resolution, shallow depth (1/8 downsampling, 128 channels).
    Captures thin rail line edges and fine ballast textures.
    """
    def __init__(self):
        super(DetailBranch, self).__init__()
        # Stage 1: 1/2 downsample
        self.s1 = nn.Sequential(
            ConvBNReLU(3, 64, ks=3, stride=2, padding=1),
            ConvBNReLU(64, 64, ks=3, stride=1, padding=1),
        )
        # Stage 2: 1/4 downsample
        self.s2 = nn.Sequential(
            ConvBNReLU(64, 64, ks=3, stride=2, padding=1),
            ConvBNReLU(64, 64, ks=3, stride=1, padding=1),
            ConvBNReLU(64, 64, ks=3, stride=1, padding=1),
        )
        # Stage 3: 1/8 downsample
        self.s3 = nn.Sequential(
            ConvBNReLU(64, 128, ks=3, stride=2, padding=1),
            ConvBNReLU(128, 128, ks=3, stride=1, padding=1),
            ConvBNReLU(128, 128, ks=3, stride=1, padding=1),
        )

    def forward(self, x):
        feat = self.s1(x)
        feat = self.s2(feat)
        feat = self.s3(feat)
        return feat


class StemBlock(nn.Module):
    """
    Stem Block of Semantic Branch (1/4 downsample).
    """
    def __init__(self):
        super(StemBlock, self).__init__()
        self.conv_in = ConvBNReLU(3, 16, ks=3, stride=2, padding=1)
        self.left = nn.Sequential(
            ConvBNReLU(16, 8, ks=1, stride=1, padding=0),
            ConvBNReLU(8, 16, ks=3, stride=2, padding=1),
        )
        self.right = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=False)
        self.fuse = ConvBNReLU(32, 16, ks=3, stride=1, padding=1)

    def forward(self, x):
        feat = self.conv_in(x)
        feat_left = self.left(feat)
        feat_right = self.right(feat)
        feat_cat = torch.cat([feat_left, feat_right], dim=1)
        return self.fuse(feat_cat)


class GELayer(nn.Module):
    """
    Gather-and-Expansion (GE) Layer for Semantic Branch.
    """
    def __init__(self, in_chan, out_chan, stride=1, exp_ratio=6):
        super(GELayer, self).__init__()
        self.stride = stride
        mid_chan = in_chan * exp_ratio

        if stride == 1:
            self.conv = nn.Sequential(
                ConvBNReLU(in_chan, in_chan, ks=3, stride=1, padding=1),
                nn.Conv2d(in_chan, mid_chan, kernel_size=3, stride=1, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(mid_chan),
                nn.ReLU(inplace=True),
                nn.Conv2d(mid_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan),
            )
        else:
            self.conv = nn.Sequential(
                ConvBNReLU(in_chan, in_chan, ks=3, stride=1, padding=1),
                nn.Conv2d(in_chan, mid_chan, kernel_size=3, stride=stride, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(mid_chan),
                nn.Conv2d(mid_chan, mid_chan, kernel_size=3, stride=1, padding=1, groups=mid_chan, bias=False),
                nn.BatchNorm2d(mid_chan),
                nn.ReLU(inplace=True),
                nn.Conv2d(mid_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan),
            )
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_chan, in_chan, kernel_size=3, stride=stride, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(in_chan),
                nn.Conv2d(in_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        if self.stride == 1:
            return self.relu(self.conv(x) + x)
        else:
            return self.relu(self.conv(x) + self.shortcut(x))


class CEBlock(nn.Module):
    """
    Context Embedding Block (Global contextual reasoning).
    """
    def __init__(self, in_chan=128, out_chan=128):
        super(CEBlock, self).__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.bn = nn.BatchNorm2d(in_chan)
        self.conv_gap = ConvBNReLU(in_chan, in_chan, ks=1, stride=1, padding=0)
        self.conv_last = ConvBNReLU(in_chan, out_chan, ks=3, stride=1, padding=1)

    def forward(self, x):
        feat = self.gap(x)
        feat = self.bn(feat)
        feat = self.conv_gap(feat)
        return self.conv_last(x + feat)


class SemanticBranch(nn.Module):
    """
    Semantic Branch: Rapid downsampling (1/32 scale) for wide contextual awareness.
    """
    def __init__(self):
        super(SemanticBranch, self).__init__()
        self.s12 = StemBlock()                       # 1/4, 16 ch
        self.s3 = nn.Sequential(
            GELayer(16, 32, stride=2),               # 1/8, 32 ch
            GELayer(32, 32, stride=1),
        )
        self.s4 = nn.Sequential(
            GELayer(32, 64, stride=2),               # 1/16, 64 ch
            GELayer(64, 64, stride=1),
        )
        self.s5 = nn.Sequential(
            GELayer(64, 128, stride=2),              # 1/32, 128 ch
            GELayer(128, 128, stride=1),
            GELayer(128, 128, stride=1),
            GELayer(128, 128, stride=1),
            CEBlock(128, 128),
        )

    def forward(self, x):
        feat2 = self.s12(x)
        feat3 = self.s3(feat2)
        feat4 = self.s4(feat3)
        feat5 = self.s5(feat4)
        return feat2, feat3, feat4, feat5


class BGALayer(nn.Module):
    """
    Bilateral Guided Aggregation (BGA) Layer.
    Fuses high-res detail cues with deep semantic features.
    """
    def __init__(self, detail_chan=128, sem_chan=128, out_chan=128):
        super(BGALayer, self).__init__()
        # Detail path
        self.detail_dw = nn.Sequential(
            nn.Conv2d(detail_chan, detail_chan, kernel_size=3, stride=1, padding=1, groups=detail_chan, bias=False),
            nn.BatchNorm2d(detail_chan),
            nn.Conv2d(detail_chan, detail_chan, kernel_size=1, stride=1, padding=0, bias=False),
        )
        self.detail_down = nn.Sequential(
            ConvBNReLU(detail_chan, detail_chan, ks=3, stride=2, padding=1),
            nn.AvgPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=False),
        )

        # Semantic path
        self.sem_dw = nn.Sequential(
            ConvBNReLU(sem_chan, sem_chan, ks=3, stride=1, padding=1),
            nn.Conv2d(sem_chan, sem_chan, kernel_size=1, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
        self.sem_up = ConvBNReLU(sem_chan, sem_chan, ks=3, stride=1, padding=1)

        self.conv_out = ConvBNReLU(detail_chan, out_chan, ks=3, stride=1, padding=1)

    def forward(self, feat_d, feat_s):
        # Detail branch outputs 1/8 scale
        # Semantic branch s5 outputs 1/32 scale -> upsample to 1/8
        d_size = feat_d.size()[2:]

        # Path 1: Detail guided by Semantic
        d_feat = self.detail_dw(feat_d)
        s_feat_up = F.interpolate(feat_s, size=d_size, mode='bilinear', align_corners=False)
        s_feat_gate = self.sem_dw(s_feat_up)
        path1 = d_feat * s_feat_gate

        # Path 2: Semantic guided by Detail
        d_feat_down = self.detail_down(feat_d)
        d_feat_gate = torch.sigmoid(d_feat_down)
        s_feat = self.sem_up(feat_s)
        s_feat_interp = F.interpolate(s_feat, size=d_feat_down.size()[2:], mode='bilinear', align_corners=False)
        path2 = s_feat_interp * d_feat_gate
        path2_up = F.interpolate(path2, size=d_size, mode='bilinear', align_corners=False)

        return self.conv_out(path1 + path2_up)


class SegmentHead(nn.Module):
    """
    Final Segmentation Prediction Head.
    """
    def __init__(self, in_chan, mid_chan, num_classes, up_factor=8):
        super(SegmentHead, self).__init__()
        self.conv = ConvBNReLU(in_chan, mid_chan, ks=3, stride=1, padding=1)
        self.drop = nn.Dropout(0.1)
        self.conv_out = nn.Conv2d(mid_chan, num_classes, kernel_size=1, stride=1, padding=0)
        self.up_factor = up_factor

    def forward(self, x, target_size=None):
        feat = self.conv(x)
        feat = self.drop(feat)
        logits = self.conv_out(feat)
        if target_size is not None:
            return F.interpolate(logits, size=target_size, mode='bilinear', align_corners=False)
        elif self.up_factor > 1:
            return F.interpolate(logits, scale_factor=self.up_factor, mode='bilinear', align_corners=False)
        return logits


class BiSeNetV2(nn.Module):
    """
    Complete BiSeNetV2 Architecture for Railway Perception (Drishti-Kavach).
    Outputs 3 classes: 0 (Background), 1 (Track_Bed), 2 (Rail_Lines).
    """
    def __init__(self, num_classes=3, is_training=True):
        super(BiSeNetV2, self).__init__()
        self.is_training = is_training
        self.detail = DetailBranch()
        self.segment = SemanticBranch()
        self.bga = BGALayer(detail_chan=128, sem_chan=128, out_chan=128)
        self.head = SegmentHead(128, 1024, num_classes, up_factor=8)

        # Auxiliary Booster Heads (Active during training for deep supervision)
        if is_training:
            self.aux2 = SegmentHead(16, 64, num_classes, up_factor=4)
            self.aux3 = SegmentHead(32, 128, num_classes, up_factor=8)
            self.aux4 = SegmentHead(64, 256, num_classes, up_factor=16)
            self.aux5 = SegmentHead(128, 512, num_classes, up_factor=32)

    def forward(self, x):
        img_size = x.size()[2:]
        feat_d = self.detail(x)
        feat2, feat3, feat4, feat5 = self.segment(x)
        feat_fuse = self.bga(feat_d, feat5)

        out = self.head(feat_fuse, target_size=img_size)

        if self.is_training and self.training:
            out_aux2 = self.aux2(feat2, target_size=img_size)
            out_aux3 = self.aux3(feat3, target_size=img_size)
            out_aux4 = self.aux4(feat4, target_size=img_size)
            out_aux5 = self.aux5(feat5, target_size=img_size)
            return out, out_aux2, out_aux3, out_aux4, out_aux5

        return out


if __name__ == "__main__":
    # Sanity check
    model = BiSeNetV2(num_classes=3, is_training=False)
    x = torch.randn(2, 3, 512, 1024)
    out = model(x)
    print("Model forward pass successful!")
    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {out.shape}")
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {params:,} (~{params/1e6:.2f}M)")



In [ ]:
# Cell 4: Joint Mixed Universal Dataset Loader
class UniversalMixedDataset(Dataset):
    def __init__(self, rs19_dir, uav_dir, split='train', img_size=(512, 1024), is_train=True):
        self.img_h, self.img_w = img_size
        self.is_train = is_train
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        
        # Load RailSem19 (Locomotive views)
        rs19_imgs = sorted(glob.glob(f'{rs19_dir}/images/{split}/*.jpg'))
        rs19_pairs = [(p, p.replace('/images/', '/masks/').replace('.jpg', '.png')) for p in rs19_imgs]
        
        # Load UAV-RSOD V1 (Indian views)
        uav_imgs = sorted(glob.glob(f'{uav_dir}/images/{split}/*.jpg'))
        uav_pairs = [(p, p.replace('/images/', '/masks/').replace('.jpg', '.png')) for p in uav_imgs]
        
        if is_train:
            # Oversample UAV pairs x5 to balance domain distribution (75% Locomotive : 25% Indian)
            balanced_uav = uav_pairs * 5
            self.all_pairs = rs19_pairs + balanced_uav
            random.seed(42)
            random.shuffle(self.all_pairs)
        else:
            self.all_pairs = rs19_pairs + uav_pairs
            
        print(f'[+] Universal {split.upper()} Dataset: {len(rs19_pairs)} Locomotive + {len(uav_pairs)} Indian Frames (Total: {len(self.all_pairs)})')
        
    def __len__(self):
        return len(self.all_pairs)
        
    def __getitem__(self, idx):
        img_p, mask_p = self.all_pairs[idx]
        img = cv2.cvtColor(cv2.imread(img_p), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_p, cv2.IMREAD_UNCHANGED)
        img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)
        if self.is_train and random.random() > 0.5:
            img = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)
        img = (img / 255.0 - self.mean) / self.std
        return torch.from_numpy(img).permute(2, 0, 1).float(), torch.from_numpy(mask).long()

class OhemCrossEntropy(nn.Module):
    def __init__(self, thresh=0.7, min_kept=50000, ignore_index=255):
        super(OhemCrossEntropy, self).__init__()
        self.thresh = float(thresh)
        self.min_kept = int(min_kept)
        self.ignore_index = ignore_index
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none')
    def forward(self, predict, target):
        b, c, h, w = predict.size()
        target = target.view(-1)
        valid_mask = target.ne(self.ignore_index)
        target = target * valid_mask.long()
        num_valid = valid_mask.sum()
        prob = F.softmax(predict, dim=1).transpose(0, 1).reshape(c, -1)
        if num_valid > 0 and self.min_kept <= num_valid:
            prob = prob.masked_fill_(~valid_mask, 1.0)
            mask_prob = prob[target, torch.arange(len(target), dtype=torch.long)]
            threshold = self.thresh
            if self.min_kept > 0:
                index = mask_prob.argsort()
                threshold_index = index[min(len(index), self.min_kept) - 1]
                if mask_prob[threshold_index] > self.thresh: threshold = mask_prob[threshold_index]
                kept_mask = mask_prob.le(threshold)
                target = target * kept_mask.long()
                valid_mask = valid_mask * kept_mask
        target = target.masked_fill_(~valid_mask, self.ignore_index).view(b, h, w)
        loss = self.criterion(predict, target)
        return loss[valid_mask.view(b, h, w)].mean()

class BiSeNetLoss(nn.Module):
    def __init__(self):
        super(BiSeNetLoss, self).__init__()
        self.crit = OhemCrossEntropy(thresh=0.7, min_kept=50000)
    def forward(self, preds, target):
        if isinstance(preds, tuple):
            main_out, aux2, aux3, aux4, aux5 = preds
            return self.crit(main_out, target) + 0.4 * (self.crit(aux2, target) + self.crit(aux3, target) + self.crit(aux4, target) + self.crit(aux5, target))
        return self.crit(preds, target)

class Evaluator:
    def __init__(self, num_classes=3):
        self.num_classes = num_classes
        self.confusion_matrix = np.zeros((num_classes, num_classes))
    def add_batch(self, gt, pred):
        mask = (gt >= 0) & (gt < self.num_classes)
        label = self.num_classes * gt[mask].astype(int) + pred[mask]
        count = np.bincount(label, minlength=self.num_classes ** 2)
        self.confusion_matrix += count.reshape(self.num_classes, self.num_classes)
    def evaluate(self):
        inter = np.diag(self.confusion_matrix)
        union = np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) - inter
        ious = inter / np.maximum(union, 1e-7)
        return np.nanmean(ious), ious
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes))


In [ ]:
# Cell 5: Universal Dual Training Loop
IMG_SIZE = (512, 1024)
BATCH_SIZE = 16
EPOCHS = 10
LR = 0.001   # Optimal joint fine-tuning rate

train_dataset = UniversalMixedDataset(RS19_DIR, UAV_DIR, split='train', img_size=IMG_SIZE, is_train=True)
val_dataset = UniversalMixedDataset(RS19_DIR, UAV_DIR, split='val', img_size=IMG_SIZE, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = BiSeNetV2(num_classes=3, is_training=True).to(device)

# Load 20-Epoch Base Weights
if os.path.exists(BASE_PTH):
    print(f'[+] Initializing from Base Checkpoint: {BASE_PTH}')
    state_dict = torch.load(BASE_PTH, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    print('[+] Checkpoint successfully loaded with 100% exact layer alignment!')
else:
    print('[!] Warning: Checkpoint not found. Training from scratch.')

criterion = BiSeNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader), eta_min=1e-6)
scaler = GradScaler()
evaluator = Evaluator(num_classes=3)

best_miou = 0.0
print('=' * 75)
print(f' 🛡️ UNIVERSAL DUAL BISENETV2 TRAINING ({EPOCHS} EPOCHS)')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Universal [{epoch:02d}/{EPOCHS}]')
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with autocast():
            preds = model(imgs)
            loss = criterion(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    model.eval()
    evaluator.reset()
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device)
            with autocast(): logits = model(imgs)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            evaluator.add_batch(masks.numpy(), preds)
            
    miou, class_ious = evaluator.evaluate()
    print(f'\n📊 Epoch [{epoch:02d}/{EPOCHS}] Results:')
    print(f' • Universal mIoU:   {miou*100:.2f}%')
    print(f'   - Background IoU: {class_ious[0]*100:.2f}%')
    print(f'   - Track Bed IoU:  {class_ious[1]*100:.2f}%')
    print(f'   - Rail Lines IoU: {class_ious[2]*100:.2f}%')
    
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), '/kaggle/working/raildrishti_seg_universal.pth')
        print(f' ⭐ NEW BEST UNIVERSAL MODEL SAVED! (mIoU: {best_miou*100:.2f}%)')


In [ ]:
# Cell 6: Export Universal Model to ONNX
print('[*] Exporting Universal Model to ONNX format...')
export_model = BiSeNetV2(num_classes=3, is_training=False).to(device)
final_pth = '/kaggle/working/raildrishti_seg_universal.pth'
if os.path.exists(final_pth):
    state_dict = torch.load(final_pth, map_location=device)
    export_model.load_state_dict(state_dict, strict=False)
    print('[+] Loaded weights with strict=False for export!')

export_model.eval()
dummy_input = torch.randn(1, 3, 512, 1024, device=device)
torch.onnx.export(
    export_model, dummy_input, '/kaggle/working/raildrishti_seg_universal.onnx',
    input_names=['images'], output_names=['output'],
    dynamic_axes={'images': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=14
)
print('[+] Successfully exported: /kaggle/working/raildrishti_seg_universal.onnx')
